# பாடம் 13 - முகவர் நினைவகம்


## அமைப்பு

இந்த நோட்புக் **Microsoft Agent Framework** (MAF) பயன்படுத்தி **நிரந்தர நினைவுடன்** பயண முன்பதிவு முகவரியை எப்படி உருவாக்குவது என்பதை காண்பிக்கிறது.

பரிமாற்றங்கள் முழுவதும் முகவரி தகவலை எவ்வாறு பேணிக் கொண்டுச்செல்கிறார் என்பதை மாறுபட்ட வகை முகவரி நினைவுகள் — வேலை நினைவகம், குறுகியகால நினைவகம் மற்றும் நீண்டகால நினைவகம் — எப்படி பாதிப்பதைக் கற்றுக்கொள்வீர்கள்.

**முன்னுதவி தேவைகள்:**
- Azure AI Foundry திட்டம் ஒன்றும், அதன் கீழ் ஒரு சொடி மாதிரி (உதாரணமாக `gpt-4o-mini`) செயல்படுத்தப்பட்டிருக்க வேண்டும்.
- Azure CLI கொண்டு உள்நுழைந்திருக்கும் — உங்கள் டெர்மினலில் `az login` இயக்கவும்.
- `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Azure AI Foundry திட்டத்தின் முடிவு இடைமுக முகவரி.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் செயல்படுத்தப்பட்ட மாதிரியின் பெயர்.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## முகவர் நினைவக வகைகள்

ஏஐ முகவர்கள் வெவ்வேறு வகையான நினைவகங்களை பயன்படுத்தலாம், ஒவ்வொன்றும் வேறுபட்ட நோக்கத்திற்காக உள்ளது:

### பணிநினைவகம்
பேச்சு துறைதான் — ஒரு அமர்வில் பரிமாறப்பட்ட செய்தಿಗಳು. முகவர் ஒரே துறையில் முன்பு உள்ள செய்திகளுக்கு மீண்டும் பார்க்க முடியும் பொருளடக்கம் தொடர வைத்திருக்க. MAF இல் இது **`agent.create_session()`** மூலம் உருவாக்கப்படுகின்றது, இது ஒரு `AgentSession` ஐ மீட்டமைக்கிறது.

### குறுகியகால நினைவகம்
ஒரு செயல்பட்டினை அல்லது அமர்வின் கால அளவை மட்டும் நீடிக்கும் தகவல் ஆனால் நிரந்தரமாக சேமிக்கப்படாது. உதாரணமாக, முகவர் பல முறுக்கு திட்டமிடல் உரையாடலின் போது உண்மைகள் சேர்க்கலாம் மற்றும் அதை இறுதி பயணத் திட்டம் தயாரிக்க பயன்படுத்தலாம்.

### நீண்டகால நினைவகம்
**அமர்வுகளுக்கு இடைவெளியில்** நீடிக்கும் விருப்பங்கள் மற்றும் உண்மைகள். மீண்டும் வரும் பயனர் தனது உணவுப் படிப்பினைகள் அல்லது பயணப் பாணியை மீண்டும் கூற வேண்டியதில்லை. நீண்டகால நினைவகம் பொதுவாக ஒரு வெளியீட்டு சேமிப்பகம் மூலம் ஆதரிக்கப்படுகிறது — தரவுத்தளம், கோப்பு, அல்லது வெக்டர் குறியீடு — மற்றும் முகவருக்கு கருவிகள் மூலம் வெளிப்படுத்தப்படுகிறது.


## அவற்றுடன் செய்யும் நினைவக செயல்பாடு

நினைவகத்தின் மிக எளிய வடிவம் உரையாடல் அமர்வு ஆகும். ஒரே அமர்வை (`agent.create_session()` மூலம் உருவாக்கப்பட்ட) தொடர்ச்சியாக `agent.run()` அழைப்புகளுக்கு வழங்கும்போது, முகவர் அந்த உரையாடலின் முழு வரலாற்றையும் பார்க்க முடியும் மற்றும் முந்தைய விவரங்களை நினைவில் கொள்ள முடியும்.

ஒரு பயண முகவரின் தொழில்நுட்பத்தை உருவாக்கி செயல்படும் நினைவகத்தை காண்பித்துみよう.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ஏஜென்ட் சரியாக பட்ஜெட்டை நினைவுக்கு கொண்டு வந்தது ஏனெனில் இரு செய்திகள் ஒரே அமர்வை பகிர்கின்றன. இது **செயல்விழிப்பு நினைவகம்** — அது அமர்வின் வாழ்நாளுக்கே மட்டுமே உள்ளது.

### புதிய திரெඩுடன் என்ன ஆகிறது?

நாம் ஒரு **புதிய** அமர்வை உருவாக்கும் போது, ஏஜென்டுக்கு முந்தைய உரையாடலின் நினைவில்லை:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## நீண்டகால நினைவுப் படிமம்

பயனர் விருப்பங்களை **அறிக்கை இடைவெளிகளுக்குள்** நினைவுகூர, உரையாடல் தொடரின் வெளியே வாழும் நிலையான ஒரு சேமிப்பிடம் தேவை. முகவர் இந்த சேமிப்பிடத்தை அணுக **கருவிகள்** மூலம் செயல்பாடுகளை அழைக்கும் மூலம் தகவல்களை சேமிக்க மற்றும் மீட்டெடுக்க முடியும்.

கீழே, ஒரு எளிய நினைவக விருப்ப சேமிப்பை (உற்பத்தியில் நீங்கள் இதை தரவுத்தளத்தோடு அல்லது வெக்டர் குறியீட்டோடு ஆதரிக்க வேண்டும்) உருவாக்கி, அதை முகவர் பயன்படுத்தக்கூடிய கருவிகளில் வெளிப்படுத்துகிறோம்.

### கட்டமைப்பு
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### நிகழ்வு 1 — முதன்முறையாக பயனர் ஆண்டு விழா பயணத்தை முன்பதிவு செய்கிறார்

சாரா முதன்முறையாக வருகிறார். முகவர் அவள் விருப்பங்களை கருவிகளின் மூலம் சேமித்து, அவற்றைப் பயன்படுத்தி ஹோட்டல்களை பரிந்துரைக்க வேண்டும்.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### நிகழ்ச்சி 2 — சில வாரங்களுக்குப் பிறகு Sarah திரும்பினாள்

Sarah ஒரு **புதிய திரையை** துவங்குகிறாள் (புதிய அமர்வை போன்றது). வேலை நினைவகம் காலியாக உள்ளது, ஆனால் நீண்டகால விருப்பக் கடை நிலுவையில் அவள் தகவல் உள்ளது. முகவர் அதை மீட்டெடுத்து பரிந்துரைகளை தனிப்பயனாக்க பயன்படுத்த வேண்டும்.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் மூன்று வகையான முகவர் நினைவகங்களை மற்றும் அவற்றை Microsoft Agent Framework உடன் எவ்வாறு செயல்படுத்துவது என்பதை கற்றுக்கொண்டீர்கள்:

| நினைவக வகை | MAF காரணிகள் | ஆயுள் |
|---|---|---|
| **பணியைச் செய்யும்** | `agent.create_session()` | ஒரு உரையாடல் |
| **குறுகிய கால** | ஒரு த்த்ரெட்டில் சேர்க்கப்பட்ட சூழல் | ஒரு பணிக்கோ அல்லது அமர்வுக்கோ |
| **நீண்ட கால** | `@tool` செயல்பாடுகள் மூலம் அணுகப்படும் வெளிப்புற சேமிப்பு | அமர்வுகளுக்கு மத்தியில் |

### முக்கியமான எடுத்துக்காட்டுகள்
1. **`agent.create_session()`** பணியாற்றும் நினைவகத்தை வழங்குகிறது — முகவர் அமர்வுக்குள் முழு உரையாடல் வரலாறを見る.
2. **புதிய அமர்வுகள் சூழலை இழக்கின்றன** — நீண்டகால நினைவகமின்றி முகவர் கடந்த உரையாடல்களை நினைவில் கொள்ள முடியாது.
3. **`@tool` செயல்பாடுகள்** இடைவெளியை இணைக்கின்றன — அவை முகவருக்கு நிலையான சேமிப்பிடத்திலிருந்து தகவல்களை சேமிக்க மற்றும் மீட்டெடுக்க அனுமதிக்கின்றன.
4. **தனிப்பயனாக்கல் நேரத்துக்கு மேல் மேம்படுகிறது** — preference கள் அதிகமாக சேமிக்கப்பட்டபோது, முகவரின் பரிந்துரைகள் சிறப்பாக இருக்கும்.

### உண்மையான உலகத்தில் பயன்பாடுகள்
- **வாடிக்கையாளர் சேவை**: வாடிக்கையாளர் வரலாறு மற்றும் விருப்பங்களை நினைவில் வைக்க
- **தனிப்பட்ட உதவியாளர்கள்**: நாட்கள் அல்லது வாரங்கள் கடந்தும் சூழலை பராமரிக்க
- **ஆரோக்கிய பராமரிப்பு**: நோயாளி தகவல்கள் மற்றும் விருப்பங்களை கண்காணிக்க
- **மின்னணு வர்த்தகம்**: வரலாற்றின் அடிப்படையில் தனிப்பட்ட சந்தைshopping

### அடுத்து செய்யவேண்டியவை
- நினைவக dict ஐ தரவுத்தளம் அல்லது வெக்டர் சேமிப்பிடம் (எ.கு. Azure AI Search) உடன் மாற்றவும்
- நேரம்-சென்சிட்டிவ் தகவலுக்கு நினைவக காலாவதியைச் சேர்க்கவும்
- பகிரப்பட்ட நினைவகத்துடன் பல முகவர் அமைப்புகளை உருவாக்கவும்
- Cognee குறிப்பேட்டைக் கொண்டு அறிவு-கிராஃப் ஆதார நினைவகத்தை ஆராயவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**தயாரிப்பு அறிவிப்பு**:  
இந்த ஆவணம் [Co-op Translator](https://github.com/Azure/co-op-translator) என்ற AI மொழிபெயர்ப்பு சேவையைப் பயன்படுத்தி மொழிபெயர்க்கப்பட்டது. தெளிவான மற்றும் துல்லியமான மொழிபெயர்ப்பிற்காக முயற்சி செய்தாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருப்பதாயிருக்கும். இயல்புநிலை மொழியில் இருந்த ஆவணம் அதிகாரப்பூர்வ மூலமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்படுத்தலில் ஏற்படும் எந்தவொரு தவறான புரிதல்களுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பില്ല என்பதை அறிவிக்கிறோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
